## إعداد الدفتر
1. احفظ `KHALEDOON_WORKER_TOKEN` و`KHALEDOON_KARMA_REF_TEXT` في Colab Secrets وفعّل **Notebook access**.
2. هذا التشغيل التجريبي مقيّد بطلب واحد محدد: ابن سينا، العربية، كارما، المقطع الأول. لا يحتاج مرجع الراوي.
3. شغّل الخلايا بالترتيب. لن يسحب العامل أي مهمة أخرى، وسيتوقف بعد هذا المقطع.
4. الناتج يذهب إلى مساحة المراجعة الخاصة فقط؛ لا يوجد نشر عام تلقائي.


# Khaledoon — VoiceTut / OmniVoice — Single Clip Pilot
تشغيل محدود وآمن لمهمة واحدة محددة في طابور Supabase. لا توجد حلقة polling مفتوحة ولا سحب عام للمهام.

الناتج يُرفع إلى مساحة المراجعة الخاصة فقط.

In [ ]:
%pip -q install git+https://github.com/k2-fsa/OmniVoice.git soundfile requests numpy
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, getpass, time, json, base64, hashlib, subprocess, threading, uuid
import numpy as np
import soundfile as sf
import requests

MODEL_ID = 'masterofaudio2077/voicetut-egyptian-tashkeel'
WORKER_URL = 'https://ypwprddkscnfblhftkeb.supabase.co/functions/v1/story-audio-worker'
from google.colab import userdata
def get_saved_secret(name, prompt):
    try:
        value = userdata.get(name)
    except Exception:
        value = ''
    return str(value or getpass.getpass(prompt)).strip()

WORKER_TOKEN = get_saved_secret('KHALEDOON_WORKER_TOKEN', 'Khaledoon worker token (if not saved in Colab Secrets): ')
WORKER_ID = 'colab-' + str(uuid.uuid4())[:8]

# مسار مرجع كارما الموجود في Google Drive وفق إعداد المشروع.
KARMA_REF_AUDIO_PATH = '/content/drive/MyDrive/Karma_Voice/assets_audio_science_magdi-yacoub_ar_000.mp3'
# اتركه فارغًا لرفع مرجع الراوي من جهازك، أو ضع مساره في Drive.
try:
    NARRATOR_REF_AUDIO_PATH = str(userdata.get('KHALEDOON_NARRATOR_REF_AUDIO_PATH') or '').strip()
except Exception:
    NARRATOR_REF_AUDIO_PATH = ''

def ensure_reference(path, label):
    if path and os.path.isfile(path):
        return path
    print('ارفع ملف مرجع ' + label + ' الصوتي الآن.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('لم يتم رفع ملف مرجع ' + label + '.')
    return '/content/' + next(iter(uploaded.keys()))

KARMA_REF_AUDIO_PATH = ensure_reference(KARMA_REF_AUDIO_PATH, 'كارما')
NARRATOR_REF_AUDIO_PATH = NARRATOR_REF_AUDIO_PATH if NARRATOR_REF_AUDIO_PATH and os.path.isfile(NARRATOR_REF_AUDIO_PATH) else ''
KARMA_REF_TEXT = get_saved_secret('KHALEDOON_KARMA_REF_TEXT', 'النص الحرفي لمرجع كارما (إذا لم يُحفظ في Colab Secrets): ')
NARRATOR_REF_TEXT = ''  # This single-job pilot uses Karma only; narrator setup is deferred.
if not WORKER_TOKEN or not KARMA_REF_TEXT:
    raise ValueError('يلزم إدخال مفتاح العامل والنص الحرفي لمرجع كارما.')
print('تم تجهيز إعدادات العامل. عند حفظ المفتاح ونص كارما في Colab Secrets لن تحتاج لإدخالهما في الجلسات التالية.')

In [ ]:
import torch
from omnivoice import OmniVoice
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print('Loading', MODEL_ID, 'on', device)
voice_model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)
print('VoiceTut model loaded.')

In [ ]:
def worker_post(payload, timeout=180):
    response = requests.post(WORKER_URL, headers={
        'Content-Type': 'application/json',
        'x-khaledoon-worker-token': WORKER_TOKEN,
    }, json=payload, timeout=timeout)
    try:
        data = response.json()
    except Exception:
        data = {'error': response.text[:1000]}
    if not response.ok:
        raise RuntimeError(f'Worker HTTP {response.status_code}: {data}')
    return data

def as_numpy_audio(result):
    sample_rate = 24000
    audio = result
    if isinstance(result, dict):
        audio = result.get('audio', result.get('waveform', result.get('samples')))
        sample_rate = int(result.get('sample_rate', result.get('sr', sample_rate)))
    elif isinstance(result, (tuple, list)) and len(result) >= 2:
        audio, sample_rate = result[0], int(result[1])
    if hasattr(audio, 'detach'):
        audio = audio.detach().cpu().numpy()
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim == 2 and audio.shape[0] <= 2 and audio.shape[1] > audio.shape[0]:
        audio = audio.T
    if audio.ndim not in (1, 2) or audio.size == 0:
        raise ValueError('Unexpected audio tensor returned by VoiceTut.')
    return audio, sample_rate

def make_mp3(job):
    payload = job.get('payload') or {}
    if job.get('speaker') != 'karma':
        raise ValueError('This pilot accepts Karma jobs only.')

    language = payload.get('language_code') or 'arz'
    if language != 'arz' or job.get('language') != 'ar':
        raise ValueError('This pilot accepts Arabic (arz) only.')

    text_to_speak = str(payload.get('text') or '').strip()
    if not text_to_speak:
        raise ValueError('Queue job has empty speech text.')

    # Match the successful local test: convert the exact reference to mono 24 kHz WAV.
    karma_wav = '/content/karma_ref_24k.wav'
    subprocess.run(
        ['ffmpeg', '-y', '-i', KARMA_REF_AUDIO_PATH, '-ac', '1', '-ar', '24000',
         '-c:a', 'pcm_s16le', karma_wav],
        check=True, capture_output=True
    )

    generated = voice_model.generate(
        text=text_to_speak,
        language='arz',
        ref_audio=karma_wav,
        ref_text=KARMA_REF_TEXT,
        num_step=36,
        guidance_scale=3.0,
        speed=0.97,
    )[0]
    audio, sample_rate = as_numpy_audio(generated)
    if audio.ndim != 1 or not np.isfinite(audio).all():
        raise ValueError('Generated audio has invalid shape or values.')
    if sample_rate != 24000:
        raise ValueError(f'Unexpected VoiceTut sample rate: {sample_rate}; expected 24000.')

    stem = f"{job['section']}-{job['slug']}-{job['language']}-{int(job['block_index']):04d}"
    wav_path, mp3_path = f'/content/{stem}.wav', f'/content/{stem}.mp3'
    sf.write(wav_path, audio, sample_rate)
    subprocess.run(
        ['ffmpeg', '-y', '-i', wav_path, '-codec:a', 'libmp3lame', '-b:a', '128k', mp3_path],
        check=True, capture_output=True
    )
    with open(mp3_path, 'rb') as f:
        mp3_bytes = f.read()
    if not mp3_bytes or len(mp3_bytes) > 13_500_000:
        raise ValueError('Generated MP3 is empty or exceeds the worker upload limit.')
    return mp3_path, mp3_bytes
print('Worker helpers ready.')

In [ ]:
TARGET_JOB_ID = '9de64804-76bc-4fbe-8ab8-3b19ecc24ff0'
EXPECTED_JOB = {
    'section': 'history',
    'slug': 'ibn-sina',
    'language': 'ar',
    'block_index': 0,
    'speaker': 'karma',
}

def run_worker():
    print('Single-job worker:', WORKER_ID)
    job = None
    heartbeat_thread = None
    lease_stop = threading.Event()
    try:
        claimed = worker_post({
            'action': 'claim',
            'job_id': TARGET_JOB_ID,
            'worker_id': WORKER_ID,
            'lease_seconds': 900,
        })
        job = claimed.get('job')
        if not job:
            print('Target job is not pending/available; no other job was claimed.')
            return

        actual = {key: job.get(key) for key in EXPECTED_JOB}
        if actual != EXPECTED_JOB or job.get('id') != TARGET_JOB_ID:
            raise RuntimeError(f'Claimed job identity mismatch; refusing generation: {actual}')

        print('Claimed exact target:', job['id'], actual)

        def heartbeat_loop(job_snapshot):
            while not lease_stop.wait(90):
                try:
                    worker_post({
                        'action': 'heartbeat',
                        'job_id': job_snapshot['id'],
                        'lock_token': job_snapshot['lock_token'],
                        'worker_id': WORKER_ID,
                    }, timeout=30)
                except Exception as heartbeat_error:
                    print('Heartbeat warning:', heartbeat_error)

        heartbeat_thread = threading.Thread(target=heartbeat_loop, args=(job,), daemon=True)
        heartbeat_thread.start()

        mp3_path, mp3_bytes = make_mp3(job)
        result = worker_post({
            'action': 'complete',
            'job_id': job['id'],
            'lock_token': job['lock_token'],
            'extension': 'mp3',
            'content_type': 'audio/mpeg',
            'audio_base64': base64.b64encode(mp3_bytes).decode('ascii'),
            'sha256': hashlib.sha256(mp3_bytes).hexdigest(),
        }, timeout=240)
        print('Private review upload:', result.get('job_id'), result.get('status'))
    except Exception as error:
        print('Single-job worker error:', repr(error))
        if job and job.get('id') and job.get('lock_token'):
            try:
                worker_post({
                    'action': 'fail',
                    'job_id': job['id'],
                    'lock_token': job['lock_token'],
                    'error': str(error)[:1800],
                }, timeout=30)
            except Exception as fail_error:
                print('Could not release failed target job:', fail_error)
        raise
    finally:
        lease_stop.set()
        if heartbeat_thread:
            heartbeat_thread.join(timeout=2)
    print('Single-job worker stopped; no further queue claim was made.')

run_worker()

## إيقاف العامل
لتوقيف حلقة العمل يدويًا، أوقف تنفيذ الخلية من Colab. الطلب الحالي سيُعاد إلى الطابور إذا انتهت مهلة الحجز، وسيُلتقط عند تشغيل العامل مرة أخرى.